In [ ]:
import os
import math
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc
import harmonypy as hm

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

DATA_DIR    = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data'
SAMPLES_DIR = os.path.join(DATA_DIR, 'samples')
OBJECTS_DIR = os.path.join(DATA_DIR, 'archive', 'lognorm_azu1_kept', 'objects')

SAMPLES = {
    'TIS04399-001': ('FH1001', 'MM Pre-Treatment'),
    'TIS02434-001': ('FH1009', 'MM Pre-Treatment'),
    'TIS02437-001': ('FH1011', 'MM Pre-Treatment'),
    'TIS04398-001': ('FH1015', 'MM Pre-Treatment'),
    'TIS02611-001': ('FH1016', 'MM Pre-Treatment'),
    'TIS02603-001': ('FH1017', 'MM Pre-Treatment'),
    'TIS02604-001': ('FH1018', 'MM Pre-Treatment'),
    'TIS02613-001': ('FH1021', 'MM Pre-Treatment'),
    'TIS02421-001': ('FH1003', 'MM Pre-Treatment'),
    'TIS02430-001': ('FH1007', 'MM Pre-Treatment'),
    'TIS02424-001': ('FH1004', 'MM Pre-Treatment'),
    'TIS02439-001': ('FH1012', 'MM Pre-Treatment'),
    'TIS02594-001': ('FH1006', 'MM Pre-Treatment'),
}

In [ ]:
objects_dir = "/home/workspace/PCD_data/xenium/fh1_pretreatment/data/archive/lognorm_azu1_kept/objects"

raw = sc.read_h5ad(os.path.join(objects_dir, "combined_raw.h5ad"))
filtered = sc.read_h5ad(os.path.join(objects_dir, "combined_filtered.h5ad"))
clustered = sc.read_h5ad(os.path.join(objects_dir, "combined_clustered.h5ad"))

In [ ]:
sc.pp.pca(filtered, n_comps=30, use_highly_variable=False, random_state=0)

ho = hm.run_harmony(
    filtered.obsm['X_pca'].astype('float64'),
    filtered.obs,
    'cohort_subject',
    random_state=0,
)

# Z_corr shape may be (n_pcs x n_cells) or (n_cells x n_pcs) depending on version
z = ho.Z_corr
if z.shape[0] != filtered.n_obs:
    z = z.T
filtered.obsm['X_pca_harmony'] = z
filtered.obsm['X_pca'] = z  # downstream steps use X_pca by default

sc.pp.neighbors(filtered, use_rep='X_pca', n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.umap(filtered, min_dist=1, spread=10, random_state=0)
sc.tl.leiden(filtered, flavor='igraph', n_iterations=2, directed=False, random_state=0)

In [ ]:
clustered.obs

In [ ]:
sc.tl.umap(filtered, min_dist=0.8, spread=4, random_state=0)
sc.tl.leiden(filtered, flavor='igraph', n_iterations=2, directed=False, random_state=0)

sc.pl.umap(
    filtered,
    color="leiden",
    legend_loc='on data',
    size=6
)

In [ ]:
# Cluster-highlight grid: the full clustering as a reference panel, then one
# panel per cluster with that cluster colored and every other cell greyed out.
HIGHLIGHT_NCOLS = 4
key = 'leiden_res0.80'

clusters = list(clustered.obs[key].cat.categories)
nrows = math.ceil((len(clusters) + 1) / HIGHLIGHT_NCOLS)
fig, axes = plt.subplots(nrows, HIGHLIGHT_NCOLS,
                         figsize=(HIGHLIGHT_NCOLS * 3, nrows * 3))
axes = axes.ravel()

sc.pl.umap(clustered, color=key, ax=axes[0], size=6,
           legend_loc="on data", legend_fontsize=6, show=False)
axes[0].set_title("all clusters")

for ax, cluster in zip(axes[1:], clusters):
    sc.pl.umap(clustered, color=key, groups=[cluster], ax=ax, size=6,
               legend_loc=None, na_in_legend=False, show=False)
    ax.set_title(f"cluster {cluster}")

for ax in axes[len(clusters) + 1:]:
    ax.axis("off")

fig.suptitle(key)
fig.tight_layout()
plt.show()